# Real captures vs RadioML: domain comparison

Descriptive only. Every cell reads `data/real_captures_2db_noise-bw.hdf5`, the synthetic subset it was
built against, and (for the raw sample histogram) the `captures/` recordings. No model or checkpoint
is loaded and nothing under a run directory is read or written, so this can run before the evaluation pass.

Each section calls one function from `scripts/analysis/domain_compare.py`. Every function takes
`(class, snr_bin)` and returns `(figure, numbers)`. **RadioML is solid blue, real is dashed orange** throughout.
Unless a section says otherwise, both domains are compared at unit power per frame, the same scaling the model sees.

In [ ]:
import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

from scripts.analysis import domain_compare as dc
from scripts.analysis.make_thesis_figures import STYLE

In [ ]:
# After the imports: the measure scripts select Agg when they load.
%matplotlib inline
import matplotlib
from IPython.display import display

matplotlib.rcParams.update(STYLE | {'figure.figsize': (10, 3.8), 'figure.dpi': 110})

CLS, BIN = 'QPSK', 16          # change freely; usable real bins below
print('real bins:', dc.real_bins())
print('synthetic reference:', dc.synthetic_path().name)


def show(result):
    fig, nums = result
    display(fig)
    w = max(map(len, nums))
    for k, v in nums.items():
        print(f'  {k:<{w}}  {v:.4g}' if isinstance(v, float) else f'  {k:<{w}}  {v}')

## 1. Averaged PSD

Hann-windowed periodograms averaged over 384 frames per domain, each scaled to unit total power. The shaded
region is the class's 99 % occupied band, read off RadioML at 30 dB so it does not widen as noise rises.
Notched real bins (spurs, all out of band) are left blank.

- `psd_rms_db` / `psd_max_db`: in-band shape mismatch after removing the mean offset (`psd_bias_db`).
  0 dB means the same spectral shape; values near 1 dB are periodogram noise.
- `psd_tilt_db`: linear fit to (real − RadioML) in dB, given as the change across the band. A non-zero
  tilt is a frequency-response slope (filter or tuner) that RadioML does not have.
- `floor_*_dbc`: median out-of-band level relative to the in-band mean per bin. The real − RadioML gap
  at the *same* bin label shows how differently the two SNR axes are defined.

In [ ]:
show(dc.psd_overlay(CLS, BIN))

## 2. PAPR and envelope

Per-frame PAPR is max |x|² / mean |x|² in dB, one value per frame. The envelope histogram pools |x| over all
frames at unit power.

- `papr_diff_db`: real minus RadioML median PAPR. A positive value means real frames are peakier. Additive
  noise, 8-bit quantization and in-band ripple all push it up, while clipping pulls it down.
- `envelope_std_*`: spread of |x| at unit power. For constant-envelope classes (FM, GMSK) any spread is added
  by the channel or receiver.

In [ ]:
show(dc.envelope_papr(CLS, BIN))

## 3. Constellation

RRC matched filter (α = 0.25, as in `make_tx`), then one sample per symbol at whichever of the 8 phases
carries the most power, per frame. Symbols are scaled to unit mean power, and the density is log-scaled.

The first figure removes each frame's estimated CFO and carrier phase. Without that step, Table I's per-frame
θ_c ~ U(0, 2π) smears any PSK into a ring, which is what the second figure shows (the model's view).

- `ring_spread_*`: std(|s|) / mean(|s|). It is 0 for ideal PSK, and noise, ISI and quantization raise it.
- `m4_coherence_*`: |E[s⁴]| / E[|s|⁴]. It is 1 for phase-locked QPSK and near 0 once the phase spins,
  so it shows whether the derotation locked.

In [ ]:
show(dc.constellation(CLS, BIN))

In [ ]:
show(dc.constellation(CLS, BIN, derotate=False))

## 4. I/Q values before normalization

The only section that does not use unit-power frames. RadioML is shown in its stored units. The real
side is rebuilt from `captures/` by `build_real_hdf5.front_end`, which reproduces the stored rows exactly
before `unit_power`, and is shown in ADC LSB. The two x-axes therefore do not share units; compare shapes.

- `rms_real` (LSB) and `adc_codes_used_real`: how much of the 8-bit range the signal exercises. A few codes
  means strong quantization, which the model never saw in RadioML.
- `dc_*`, `iq_gain_ratio_*_db`, `iq_corr_*`: DC offset (per unit RMS), I/Q gain imbalance and I–Q correlation
  (≈ phase imbalance). All ≈ 0 for a balanced receiver.
- `kurtosis_*`: excess kurtosis per channel. 0 = Gaussian, negative = bounded/constellation-like.
- `frame_power_spread_*_db`: std of per-frame RMS, which is exactly what `unit_power` removes. A bin fed by
  two capture rungs shows up here as two peaks.

In [ ]:
show(dc.sample_histogram(CLS, BIN))

## 5. Spectrogram of one frame

64-sample Hann STFT with 75 % overlap, on a shared dB scale (frame total = 0 dB, darker = more power).
`frame` picks which of the evenly spaced frames is drawn.

- `inband_power_std_*_db`: how much the in-band power moves over the 1 ms frame. Fading (RadioML's
  Table I channel H) or AGC/gain steps raise it, and for a stationary signal it is set by STFT variance alone.

In [ ]:
show(dc.spectrogram(CLS, BIN, frame=0))

## 6. Out-of-band noise colour

Averaged PSD outside the occupied band ± 30 kHz, with the notched bins removed. Each domain is referenced to
its own out-of-band median, so a flat line at 0 dB means white noise. Dotted lines are the linear fits.

- `noise_tilt_*_db`: change of the fitted line across the out-of-band span.
- `noise_edge_droop_*_db`: outer 10 % of each band edge relative to the rest. Negative means the
  anti-alias filter is rolling off, and positive means the floor lifts at the edges.
- `noise_ripple_*_db`: rms deviation from the linear fit. Around 0.2 dB is periodogram noise, and more
  than that is real structure, such as a dip or bump in the receiver's response.

In [ ]:
show(dc.noise_color(CLS, BIN))

## 7. Carrier frequency offset

One estimate per frame. Where the class has a spectral line at `order`×CFO, it is the peak of the
`x**order` spectrum divided by `order`: 1 for ASK/OOK/AM-SSB, 2 for BPSK/AM-DSB, 4 for QPSK/QAM/APSK, 8 for 8PSK.
Otherwise it is the power-weighted spectral centroid of the band, which also picks up spectral asymmetry and
in-band tilt, so it is not a clean CFO. That covers 16/32PSK, FM, GMSK and OQPSK.

Table I of O'Shea et al. (2018) gives Δf_c ~ N(0, σ_clk) but no value for σ_clk. The dotted curve is that
form with σ_clk fitted robustly (1.4826·MAD) to the RadioML estimates, so it shows the stated shape, not a published σ.

- `cfo_median_*_hz`: the typical offset. A constant real offset is the mismatch between the HackRF and RTL-SDR
  reference oscillators, which RadioML's zero-mean draw does not model.
- `cfo_iqr_*_hz`, `cfo_reliable_*`: spread across frames. An IQR above 1 kHz means the estimator found no line
  (e.g. high-order PSK on few ADC codes), and the table brackets those medians.
- `cfo_ks`: two-sample Kolmogorov–Smirnov distance between the domains' CFO distributions (0 = identical,
  1 = disjoint).

In [ ]:
show(dc.cfo_estimate(CLS, BIN))

In [ ]:
show(dc.cfo_estimate('FM', BIN))   # centroid: RadioML FM sits +32 kHz by construction

## 8. ADC loading

How much of the 8-bit range each domain uses, per class × bin. The real side reads the raw RTL-SDR codes
(uint8 − 127.5, before shift and notch). The comparison is the synthetic `quantization` condition, whose full
scale per frame is the 99.9th percentile of |I| and |Q|. Its frames are re-quantized from the subset here and
must match the condition file code for code, which proves the rows line up. Both quantizers are 8-bit
mid-rise with full scale = 128 LSB, so every column shares units. All values are medians over frames.

- `rms_lsb_*` / `peak_lsb_*`: per-branch RMS and peak. The synthetic peak is 127.5 by construction.
- `headroom_*_db`: 20·log10(128 / 99.9th-percentile level), i.e. how far below full scale the signal sits.
  It is 0 dB for the synthetic condition by definition.
- `codes_*`: distinct codes per frame, I and Q pooled.
- `enob_*_bits`: the bits an ideal ADC driven by a full-scale sine would need for the same SQNR,
  (10·log10(σ² / (Δ²/12)) − 1.76) / 6.02. It is 8.0 for a full-scale sine. It measures loading only;
  for the real side σ includes receiver noise.
- `sqnr_inband_*_db`: in-band signal (band PSD minus the out-of-band floor) over quantization noise
  Δ²/6 (I+Q, Δ = 1 LSB) spread evenly over 1.024 MHz, counting only the occupied-band share.
- `qerr_model_ratio_synth`: measured / modelled quantization error on the synthetic side. Values above 1 come
  from the 0.1 % of samples its percentile reference clips.

The full table (every class × usable bin) is written by `make_thesis_figures.py` to
`outputs/figures/adc_loading.csv`. The cell below computes a few classes.

In [ ]:
adc = dc.adc_loading_table(classes=('QPSK', '16QAM', 'FM', 'AM-DSB-SC', 'OOK'))
print(dc.format_adc_table(adc))

## 9. CFO-corrected diagnostic set

`data/real_captures_2db_noise-bw_cfodc.hdf5`, built by `scripts/analysis/build_real_cfodc.py`, holds the
**same** rows, labels and bins as the frozen real set, with each frame derotated by its capture's CFO. It is a
DIAGNOSTIC set: it is evaluated separately to ask how much of the gap is CFO, and it never replaces the primary set.

Each capture's CFO comes from its |FFT(xᵖ)|² summed over the whole recording (p ∈ {1, 2, 4, 8}, best line
≥ 3 dB above the floor). Frames whose own estimate locks within 300 Hz of that line are derotated per frame
(when ≥ 90 % of a capture's frames lock), and the rest get the capture value. Captures with no line
(high-order PSK, FM, weak lines at low SNR) take the median of the 4 line-bearing captures nearest in capture
time. The per-frame centroid is not used: it reads ~2 kHz of in-band tilt, plus FM's intended +32 kHz, as CFO.

- `leave_one_out_error_hz`: how well time neighbours predict a capture whose line *is* known. This is the
  expected error of the no-line fallback.
- `residual_hz`: the line left after derotation, per line-bearing capture. It should be near 0.

In [ ]:
import json, h5py
import numpy as np

CFODC = 'data/real_captures_2db_noise-bw_cfodc.hdf5'
with h5py.File(CFODC, 'r') as f:
    print(f.attrs['diagnostic'])
    diag = json.loads(f.attrs['metadata'])['diagnostic']
print(json.dumps(diag['counts'], indent=1))
print('leave-one-out |error| Hz:', diag['method']['leave_one_out_error_hz'])
res = [abs(r['residual_hz']) for r in diag['captures'] if 'residual_hz' in r]
print(f'residual line |Hz|: median {np.median(res):.0f}, max {np.max(res):.0f}')

In [ ]:
# Per-frame CFO of the corrected set vs RadioML: line classes should now sit near 0 Hz.
# (Carrier phase is still random per frame, as in RadioML, so constellations still need derotate=True.)
CFO_SRC = dc.Sources(real=CFODC)
show(dc.cfo_estimate(CLS, BIN, src=CFO_SRC))
show(dc.cfo_estimate("AM-DSB-SC", 0, src=CFO_SRC))

## Aggregate table

One row per class at the chosen bin(s), computed from a single load of each cell with no figures.
This is the section's main quantitative artifact. The thesis version is written by
`scripts/analysis/make_thesis_figures.py` to `outputs/figures/domain_aggregate.csv`.

Columns: PSD rms/max/tilt mismatch (§1), PAPR difference (§2), median CFO per domain (§7, bracketed = no line
found), noise-floor tilt per domain (§6), and the out-of-band floor per domain (§1, SNR-definition gap).

In [ ]:
TABLE_BINS = (16,)
rows = dc.aggregate_table(TABLE_BINS)
print(dc.format_table(rows))

In [ ]:
# Optional: keep a notebook copy next to the thesis one.
# dc.write_csv(rows, 'outputs/figures/domain_aggregate_notebook.csv')